# Masked-Diffusion BabyLM — Strict-Small (Evaluation)

A masked-diffusion denoiser is scored like a masked LM (per-token
pseudo-log-likelihood), so we evaluate it with the official pipeline's **`mlm`**
backend. We run:

1. Full zero-shot + GLUE fine-tuning on the **final** model (`main`).
2. Fast zero-shot on **every** `chck_NM` checkpoint.
3. `collate_preds` to build the submission file.

The model must already be public on the Hub (see `2_upload_pipeline.ipynb`).

In [ ]:
# Cell 1 — Parameters
# Set MODEL_ID MANUALLY to the Hub repo you want to evaluate. The next cell lists
# your public models on the Hub (huggingface.co/amosluna) so you can copy one here.
MODEL_ID = "amosluna/babylm-2026-strict-small-mdlm-seed42"
BACKEND  = "mlm"            # masked-diffusion is scored as a masked LM
TRACK    = "strict-small"
EVAL_REPO = "https://github.com/Amos-Luna/Masked-Diffusion-BabyLM.git"
DRIVE_EVAL_ROOT = "/content/drive/MyDrive/Researchs/BabyLM_diffusion_G4/results"  # where results are persisted

In [ ]:
# Cell 2 — Setup: mount Drive, clone repo, install eval deps, HF token
import os, subprocess
# Quiet the noise. These env vars are inherited by every `!python -m ...`
# subprocess below, so they silence: TensorFlow oneDNN/CPU banners, protobuf
# "gencode version" UserWarnings, HF download progress bars, the "A new version
# of model.py was downloaded" notices, and tokenizer fork warnings. Only real
# output (eval scores, errors) remains.
os.environ.update({
    "TF_CPP_MIN_LOG_LEVEL": "3",          # hide TensorFlow INFO/WARNING banners
    "TF_ENABLE_ONEDNN_OPTS": "0",         # drop the oneDNN notice
    "TRANSFORMERS_VERBOSITY": "error",    # mute "A new version ... was downloaded"
    "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",  # no file-download bars
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONWARNINGS": "ignore",           # silence protobuf/other UserWarnings
})
from google.colab import drive; drive.mount("/content/drive")
if not os.path.isdir("/content/Masked-Diffusion-BabyLM"):
    subprocess.run(["git", "clone", EVAL_REPO, "/content/Masked-Diffusion-BabyLM"], check=True)
%cd /content/Masked-Diffusion-BabyLM/strict
!pip install -q --progress-bar off -r requirements.txt
# requirements.txt pins torch==2.7.0, but Colab ships a torchvision built for a
# newer torch (0.26.0 -> torch 2.11). The mismatch makes torchvision's compiled
# ops fail to register, and `transformers` crashes when it lazily imports
# torchvision via AutoProcessor (RuntimeError: operator torchvision::nms ...).
# This eval is TEXT-only and never needs torchvision, so we remove it; with it
# absent, transformers simply skips the vision path. (Alternative: install the
# matching torchvision==0.22.0 for torch 2.7.0.)
!pip uninstall -y -q torchvision || true
!python -m scripts.download_evals   # downloads BLiMP/COMPS/entity_tracking/GLUE + fast EWoK
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets.", e)
# EWoK *full* eval is a gated dataset (ewok-core/ewok-core-1.0), so it is NOT in
# the snapshot above and must be downloaded + vocab-filtered here. Requires:
#   (a) accepting the terms at huggingface.co/datasets/ewok-core/ewok-core-1.0
#       with THIS HF account (the HF_TOKEN set just above), and
#   (b) the nltk 'punkt' tokenizer. Best-effort: if access isn't granted, EWoK is
#   simply scored as None and the rest of the zero-shot suite still runs.
try:
    import nltk
    nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
    subprocess.run(["python", "-m", "evaluation_pipeline.ewok.dl_and_filter"], check=True)
    print("EWoK full-eval data ready -> evaluation_data/full_eval/ewok_filtered")
except Exception as e:
    print("Skipping EWoK full-eval (gated dataset not accessible):", e)

In [ ]:
# Cell 3 — List your models + their checkpoint revisions on the Hub (manual pick)
# Lists every model under huggingface.co/amosluna and the chck_NM revisions each
# one exposes, so you can copy the right repo into MODEL_ID (Cell 1) by hand.
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
print("Models on huggingface.co/amosluna:\n")
for m in api.list_models(author="amosluna"):
    try:
        revs = sorted(r.name for r in api.list_repo_refs(m.id).branches if r.name.startswith("chck_"))
    except Exception:
        revs = []
    print(f"  {m.id}")
    if revs:
        print(f"      revisions: {', '.join(revs)}")
print(f"\n>> Currently selected MODEL_ID: {MODEL_ID}")

In [ ]:
# Cell 4 — One-time fix: make the uploaded repo loadable by the eval pipeline.
# Patches main + every chck_* revision (tiny text files only; weights are NOT
# re-uploaded, so it runs in seconds). Two fixes:
#   (1) tokenizer_config.json -> tokenizer_class="PreTrainedTokenizerFast"
#       (models saved by transformers>=5 use "TokenizersBackend", which the eval
#        pipeline's transformers 4.51.x can't resolve -> AutoProcessor error).
#   (2) re-pushes the latest model.py/config.py so trust_remote_code fixes reach
#       old checkpoints (e.g. forward() now tolerates token_type_ids, which the
#       official `reading` task passes -> was crashing with a TypeError).
# Idempotent and safe to re-run. Requires a WRITE HF token (set in Cell 2).
%cd /content/Masked-Diffusion-BabyLM/diffusion
!python scripts/fix_hub_for_eval.py --repo-id $MODEL_ID
%cd /content/Masked-Diffusion-BabyLM/strict

In [ ]:
# Cell 5 — FULL zero-shot on the final checkpoint (main)
# Scripts live in strict/scripts/ but read data from strict/ (relative paths),
# so we run them from strict/ as `bash scripts/<name>.sh`.
!bash scripts/eval_zero_shot.sh $MODEL_ID $BACKEND

In [ ]:
# Cell 6 — FAST zero-shot on all chck_NM checkpoints (intermediate checkpoints)
# Loops chck_1M..chck_10M then chck_20M..chck_100M (strict-small stops at 100M).
!bash scripts/eval_zero_shot_fast_all_revisions.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 7 — GLUE fine-tuning on the final checkpoint (full eval requirement)
# Uses named args; writes per-task scores under strict/results/.
!bash scripts/eval_finetuning.sh --model_path $MODEL_ID --seed 42

In [ ]:
# Cell 8 — (Optional) diffusion-native scorer: ELBO and the layer-duplication
# variant, which the official pipeline cannot express. Writes predictions.json
# in the same layout so it is interchangeable with the mlm backend above.
%cd /content/Masked-Diffusion-BabyLM/diffusion
!python scripts/diffusion_eval_backend.py --model_path_or_name $MODEL_ID \
    --task blimp --data_path ../strict/evaluation_data/full_eval/blimp_filtered \
    --scoring elbo --layer_duplication_factor 2 --backend mlm --save_predictions

In [ ]:
# Cell 9 — Build the submission file (predictions scored server-side)
# collate_preds.sh already passes --fast (required for a full Challenge submission).
%cd /content/Masked-Diffusion-BabyLM/strict
!bash scripts/collate_preds.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 10 — Collect results into a CSV + persist everything to Drive
# strict/results/ is on the ephemeral Colab disk, so we (1) print the markdown
# table, (2) flatten scores into results_summary.csv, and (3) copy results/ +
# the submission zip to Drive so they survive disconnects and feed the paper.
import csv, glob, os, shutil, subprocess
%cd /content/Masked-Diffusion-BabyLM/strict

print(subprocess.run(["python", "scripts/print_results_table.py"],
                     capture_output=True, text=True).stdout)

rows = []  # (split, task, subtask_or_metric, score)
# Reports live at results/<model>/main/zero_shot/<backend>/<task>/<dataset>/...
# (backend is mlm here, NOT causal).
for rep in sorted(glob.glob(f"results/*/main/zero_shot/{BACKEND}/*/*/best_temperature_report.txt")):
    p = rep.split("/")
    for i, line in enumerate(open(rep)):
        if line.strip().startswith("### AVERAGE"):
            try: rows.append(("zero_shot", p[-3], p[-2], float(open(rep).read().splitlines()[i+1].strip())))
            except Exception: pass
            break
for res in sorted(glob.glob("results/*/main/finetune/*/results.txt")):
    task = res.split("/")[-2]
    for line in open(res):
        k, _, v = line.partition(":")
        if k.strip() in ("accuracy", "f1", "mcc"):
            try: rows.append(("finetune", task, k.strip(), float(v.strip()) * 100))
            except Exception: pass

safe = MODEL_ID.replace("/", "__")
dst = f"{DRIVE_EVAL_ROOT}/{safe}"
os.makedirs(dst, exist_ok=True)
with open(f"{dst}/results_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["split", "task", "metric", "score"]); w.writerows(rows)
if os.path.isdir("results"):
    shutil.copytree("results", f"{dst}/results", dirs_exist_ok=True)
for z in glob.glob("*.zip"):  # collate_preds writes the submission zip here
    shutil.copy2(z, dst)
print(f"\nSaved {len(rows)} scores -> {dst}/results_summary.csv (+ results/ and submission on Drive)")